# Analyze results for reranker optimization

In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
from IPython.display import display

In [ ]:
pd.options.plotting.backend = "plotly"

create_images = True

baseline_runs = [
    "", "", "" # enter results files to analyse
]

comparing_runs = [
    "", "", "" # enter results files to analyse
]

metrics = [
    "context_precision","context_recall","nv_context_relevance","answer_relevancy",
    "nv_response_groundedness", "correctness","semantic_similarity",
    "trustworthiness","prompt_alignment"
]

metrics_to_compare = [
    "context_precision", "nv_context_relevance", "context_recall", "correctness"
]

Flatten and combine all results from all runs into one dataframe

In [ ]:
baseline_data = []

for run_file in baseline_runs:
    with open(run_file, 'r') as f:
        run_data = json.load(f)
        run_name = f.name.split('/')[-1]
        for entry in run_data['results']:
            dataset_name = entry["dataset"]
            
            for result in entry["results"]:
                r = result.copy()
                r["dataset"] = dataset_name
                r["run"] = run_name
                baseline_data.append(r)

baseline_df = pd.DataFrame(baseline_data)

baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'context_precision'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'context_recall'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'answer_relevancy'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'semantic_similarity'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'trustworthiness'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'nv_response_groundedness'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'multiple choice', 'answer_relevancy'] = float('nan')

In [ ]:
comparing_data = []

for run_file in comparing_runs:
    with open(run_file, 'r') as f:
        run_data = json.load(f)
        run_name = f.name.split('/')[-1]
        for entry in run_data['results']:
            dataset_name = entry["dataset"]
            
            for result in entry["results"]:
                r = result.copy()
                r["dataset"] = dataset_name
                r["run"] = run_name
                comparing_data.append(r)

comparing_df = pd.DataFrame(comparing_data)

comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'context_precision'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'context_recall'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'answer_relevancy'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'semantic_similarity'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'trustworthiness'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'nv_response_groundedness'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'multiple choice', 'answer_relevancy'] = float('nan')

In [ ]:
baseline_detail = baseline_df.groupby('dataset')[metrics].mean()
baseline_by_metric = baseline_df[metrics_to_compare].mean()

comparing_detail = comparing_df.groupby('dataset')[metrics].mean()
comparing_by_metric = comparing_df[metrics_to_compare].mean()

baseline_by_dataset = baseline_df.groupby('dataset')[metrics_to_compare].mean().T.mean()
comparing_by_dataset = comparing_df.groupby('dataset')[metrics_to_compare].mean().T.mean()

baseline_by_run = baseline_df.groupby('run')[metrics_to_compare].mean()
comparing_by_run = comparing_df.groupby('run')[metrics_to_compare].mean()

## Visualizing results

In [ ]:
baseline_overall = baseline_df[metrics].mean().mean()
comparing_overall = comparing_df[metrics].mean().mean()

fig = go.Figure(go.Indicator(
    mode = "number+delta",
    value = np.round(comparing_overall,3),
    number = {'prefix': f"Overall changed from {np.round(baseline_overall,3)} to "},
    delta = {'position': "top", 'reference': np.round(baseline_overall,3)}
))

fig.show()

if create_images:
    pio.write_image(fig, 'overall-delta.pdf')

In [ ]:
fig = go.Figure()

baseline_text = np.round(baseline_by_metric.values.tolist(), 2)
comparing_text = np.round(comparing_by_metric.values.tolist(), 2)

fig.add_trace(
    go.Bar(
        x=baseline_by_metric.index.tolist(),
        y=baseline_by_metric.values.tolist(),
        name="baseline",
        marker_color="grey",
        text=baseline_text
    )
)

fig.add_trace(
    go.Bar(
        x=baseline_by_metric.index.tolist(),
        y=comparing_by_metric.values.tolist(),
        name="comparing",
        marker_color='#0EA4E3',
        text=comparing_text
    )
)

fig.update_layout(
    barmode='group',
    yaxis=dict(
        range=[0, 1],
        title="Mean score"
    ),
    xaxis=dict(
        title="Metric"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'metric-comparison-bar.pdf')

In [ ]:
fig = go.Figure()

baseline_text = np.round(baseline_by_dataset.values.tolist(), 2)
comparing_text = np.round(comparing_by_dataset.values.tolist(), 2)

fig.add_trace(
    go.Bar(
        x=baseline_by_dataset.index.tolist(),
        y=baseline_by_dataset.values.tolist(),
        name="baseline",
        marker_color="grey",
        text=baseline_text
    )
)

fig.add_trace(
    go.Bar(
        x=baseline_by_dataset.index.tolist(),
        y=comparing_by_dataset.values.tolist(),
        name="comparing",
        marker_color='#0EA4E3',
        text=comparing_text
    )
)


fig.update_layout(
    barmode='group',
    yaxis=dict(
        range=[0, 1],
        title="Mean score"
    ),
    xaxis=dict(
        title="Dataset"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'dataset-comparison-bar.pdf')

In [ ]:
metrics_diffs = comparing_by_metric - baseline_by_metric
metrics_percentage_diff = np.round(metrics_diffs / baseline_by_metric * 100, 2)

fig = go.Figure()

for metric in metrics_diffs.index:
    y = metrics_diffs[metric]
    color = "#FF7B52" if y <= 0.0 else "#56D752"
    fig.add_trace(
        go.Bar(x=[metric], y=[y], marker_color=color, text=f"{np.round(y, 4)}<br>({metrics_percentage_diff[metric]}%)")
    )

fig.update_layout(
    yaxis=dict(
        range=[-0.45, 0.45],
        title="Delta"
    ),
    xaxis=dict(
        title=""
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'metrics-delta-bar.pdf')

In [ ]:
dataset_diffs = comparing_by_dataset - baseline_by_dataset
dataset_percentage_diffs = np.round(dataset_diffs / baseline_by_dataset * 100, 2)

fig = go.Figure()

for dataset in dataset_diffs.index:
    y = dataset_diffs[dataset]
    color = "#FF7B52" if y <= 0.0 else "#56D752"
    fig.add_trace(
        go.Bar(x=[dataset], y=[y], marker_color=color, text=f"{np.round(y, 4)}<br>({dataset_percentage_diffs[dataset]}%)")
    )

fig.update_layout(
    yaxis=dict(
        range=[-0.4, 0.4],
        title="Delta"
    ),
    xaxis=dict(
        title="Dataset"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'dataset-delta-bar.pdf')

In [ ]:
detail = comparing_df.groupby('dataset')[metrics].mean()

text_matrix = detail.round(2).astype(str)
text_matrix = text_matrix.replace("nan", "n.a.")

deatil_fig = go.Figure(data=go.Heatmap(
    z=detail,
    x=detail.columns,
    y=detail.index,
    text=text_matrix,
    zmin=0.0,
    zmax=1.0,
    texttemplate="%{text}",
    colorscale=[(0.00, "#FF7B52"), (0.5, "#FF7B52"), 
                (0.5, "#FFC44F"), (0.7, "#FFC44F"),
                (0.7, "#F1FF53"), (0.9, "#F1FF53"),
                (0.9, "#56D752"),  (1.00, "#56D752")]
))

deatil_fig.update_layout(
    xaxis_title="Metrics",
    yaxis_title="Datasets",
    title="Mean score per dataset and metric",
    xaxis_nticks=len(detail.columns),
    yaxis_nticks=len(detail.index),
    font=dict(
        size=16,
    ),

)
    
deatil_fig.show()

if create_images:
    pio.write_image(deatil_fig, 'detail-scores.png', scale=5)

In [ ]:
heatmap_data = comparing_detail - baseline_detail

text_matrix = heatmap_data.round(2).astype(str)
text_matrix = text_matrix.replace("nan", "n.a.")

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    text=text_matrix,
    texttemplate="%{text}",
    colorscale=[(0.00, "#FF7B52"), (0.5, "#F1FF53"), (1.00, "#56D752")],
))

fig.update_layout(
    xaxis_title="Metrics",
    yaxis_title="Datasets",
    title="Mean delta per dataset and metric",
    xaxis_nticks=len(detail.columns),
    yaxis_nticks=len(detail.index),
    font=dict(
        size=16,
    ),

)

fig.show()

if create_images:
    pio.write_image(fig, 'detail-delta.png', scale=5)

In [ ]:
results_by_metric = comparing_df[metrics].mean()

mt_means_fig = go.Figure()

for metric in results_by_metric.index:
    y = results_by_metric[metric]
    color = "#FF7B52" if y <= 0.5 else "#FFC44F" if (y <= 0.7 and y > 0.5) else "#F1FF53" if (y <= 0.9 and y > 0.7) else "#56D752"
    mt_means_fig.add_trace(
        go.Bar(x=[metric], y=[results_by_metric[metric]], marker_color=color, text=np.round(y, 2))
    )

mt_means_fig.update_layout(
    xaxis_title='Metric',
    yaxis_title='Mean score',
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    ),
)

mt_means_fig.show()

if create_images:
    pio.write_image(mt_means_fig, 'metric-means-bar.pdf')

In [ ]:
baseline_alignment = baseline_df['prompt_alignment'].mean()
comparing_alignment = comparing_df['prompt_alignment'].mean()

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=["baseline"],
        y=[baseline_alignment],
        name="baseline",
        marker_color="grey",
        text=[np.round(baseline_alignment, 2)]
    )
)

fig.add_trace(
    go.Bar(
        x=["comparing"],
        y=[comparing_alignment],
        name="comparing",
        marker_color='#0EA4E3',
        text=[np.round(comparing_alignment, 2)]
    )
)

fig.update_layout(
    barmode='group',
    yaxis=dict(
        range=[0, 1],
        title="Mean score"
    ),
    xaxis=dict(
        title="prompt_alignment"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

Manually rerank chunks with the model to visualize the relevance over all 4 chunks. The goal is to check if the context_precision could not be improved for general QA and multi-hop QA because all relevant chunks already were in the context window, so that the context compression by the reranker had no effekt.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("nvidia/llama-nemotron-rerank-1b-v2", trust_remote_code=True, dtype="auto")
tokenizer = AutoTokenizer.from_pretrained("nvidia/llama-nemotron-rerank-1b-v2", trust_remote_code=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def score_chunks(query, chunks):
    pairs = [[query, passage] for passage in chunks]
    inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors="pt", max_length=10240).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        scores = outputs.logits.view(-1, ).float().cpu().tolist()
    
    return pd.DataFrame({'score': scores, 'chunk': chunks})

def get_mean_relevance_scores(dataframe: pd.DataFrame, dataset):
    dataset_tasks = dataframe[dataframe['dataset']==dataset]

    scores = []

    for _, task in dataset_tasks.iterrows():
        query = task['user_input']
        chunks = task['retrieved_contexts']
        result = score_chunks(query, chunks)
        scores.append(result['score'].values)
    
    max_len = max(len(sublist) for sublist in scores)
    padded = np.array([np.pad(sublist, (0, max_len - len(sublist)), constant_values=np.nan) for sublist in scores], dtype=float)
    scores = np.array(padded)

    mean_relevance_scores = []
    for i in range(max_len):
        mean_relevance_scores.append(np.nanmean(scores[:, i]))

    return np.array(mean_relevance_scores)

In [ ]:
baseline_norm_mc_relevances = get_mean_relevance_scores(baseline_df, 'multiple choice')
baseline_norm_mh_relevances = get_mean_relevance_scores(baseline_df, 'multi-hop QA')
baseline_norm_gqa_relevances = get_mean_relevance_scores(baseline_df, 'general QA')
baseline_norm_fc_relevances = get_mean_relevance_scores(baseline_df, 'fact checking')
baseline_norm_cqa_relevances = get_mean_relevance_scores(baseline_df, 'corporate QA')

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(len(baseline_norm_mc_relevances))+1, y=baseline_norm_mc_relevances, mode='lines+markers', name='multiple choice', line=dict(color='#BF4F51', width=3)))
fig.add_trace(go.Scatter(x=np.arange(len(baseline_norm_mh_relevances))+1, y=baseline_norm_mh_relevances, mode='lines+markers', name='multi-hop QA', line=dict(color='#FFBF00', width=3)))
fig.add_trace(go.Scatter(x=np.arange(len(baseline_norm_gqa_relevances))+1, y=baseline_norm_gqa_relevances, mode='lines+markers', name='general QA', line=dict(color='#6CB4EE', width=3)))
fig.add_trace(go.Scatter(x=np.arange(len(baseline_norm_fc_relevances))+1, y=baseline_norm_fc_relevances, mode='lines+markers', name='fact-checking', line=dict(color='#8DB600', width=3)))
fig.add_trace(go.Scatter(x=np.arange(len(baseline_norm_cqa_relevances))+1, y=baseline_norm_cqa_relevances, mode='lines+markers', name='corporage QA', line=dict(color='#B284BE', width=3)))
fig.update_layout(
    title='Mean relevance scores for chunks at position x<br>(before optimization)', 
    xaxis_title='Chunk position', 
    yaxis_title='Cross-encoder score',
    height=500,
    width=1000,
    yaxis=dict(
        range=[-13, 5],
    ),
    font=dict(
        size=16,
    ))
fig.show()

if create_images:
    pio.write_image(fig, 'relevance-scores-baseline.pdf')


comparing_norm_mc_relevances = get_mean_relevance_scores(comparing_df, 'multiple choice')
comparing_norm_mh_relevances = get_mean_relevance_scores(comparing_df, 'multi-hop QA')
comparing_norm_mh_relevances = get_mean_relevance_scores(comparing_df, 'multi-hop QA')
comparing_norm_gqa_relevances = get_mean_relevance_scores(comparing_df, 'general QA')
comparing_norm_fc_relevances = get_mean_relevance_scores(comparing_df, 'fact checking')
comparing_norm_cqa_relevances = get_mean_relevance_scores(comparing_df, 'corporate QA')

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(len(comparing_norm_mc_relevances))+1, y=comparing_norm_mc_relevances, mode='lines+markers', name='multiple choice', line=dict(color='#BF4F51', width=3)))
fig.add_trace(go.Scatter(x=np.arange(len(comparing_norm_mh_relevances))+1, y=comparing_norm_mh_relevances, mode='lines+markers', name='multi-hop QA', line=dict(color='#FFBF00', width=3)))
fig.add_trace(go.Scatter(x=np.arange(len(comparing_norm_gqa_relevances))+1, y=comparing_norm_gqa_relevances, mode='lines+markers', name='general QA', line=dict(color='#6CB4EE', width=3)))
fig.add_trace(go.Scatter(x=np.arange(len(comparing_norm_fc_relevances))+1, y=comparing_norm_fc_relevances, mode='lines+markers', name='fact checking', line=dict(color='#8DB600', width=3)))
fig.add_trace(go.Scatter(x=np.arange(len(comparing_norm_cqa_relevances))+1, y=comparing_norm_cqa_relevances, mode='lines+markers', name='corporage QA', line=dict(color='#B284BE', width=3)))
fig.update_layout(
    title='Mean relevance scores for chunks at position x<br>(after optimization)', 
    xaxis_title='Chunk position', 
    yaxis_title='Cross-encoder score',
    height=500,
    width=600,
    yaxis=dict(
        range=[-13, 5],
    ),
    font=dict(
        size=16,
    ))
fig.show()

if create_images:
    pio.write_image(fig, 'relevance-scores-comparing.pdf')

In [ ]:
print("mean relevance for the first 4 chunks (before optimization)")

print(f"multi-hop QA {baseline_norm_mh_relevances[:4].mean()}")
print(f"general QA {baseline_norm_gqa_relevances[:4].mean()}")
print(f"corporate QA {baseline_norm_cqa_relevances[:4].mean()}")


print("mean relevance for the first 4 chunks (after optimization)")

print(f"multi-hop QA {comparing_norm_mh_relevances[:4].mean()}")
print(f"general QA {comparing_norm_gqa_relevances[:4].mean()}")
print(f"corporate QA {comparing_norm_cqa_relevances[:4].mean()}")

In [ ]:
baseline_relevances = np.array([baseline_norm_mh_relevances[:4].mean(), baseline_norm_gqa_relevances[:4].mean(), baseline_norm_cqa_relevances[:4].mean()]).T
comparing_relevances = np.array([comparing_norm_mh_relevances[:4].mean(), comparing_norm_gqa_relevances[:4].mean(), comparing_norm_cqa_relevances[:4].mean()]).T

relevance_diffs = comparing_relevances - baseline_relevances 

relevance_percentage_diffs = np.round(relevance_diffs / np.abs(baseline_relevances) * 100, 2)

x = ["multi-hop QA", "general QA", "corporate QA"]

fig = go.Figure()

for i, r in enumerate(relevance_diffs):
    y = r
    color = "#FF7B52" if y <= 0.0 else "#56D752"
    fig.add_trace(
        go.Bar(x=[x[i]], y=[y], marker_color=color, text=f"{np.round(y, 4)}<br>({relevance_percentage_diffs[i]}%)")
    )

fig.update_layout(
    title="Delta of the cross-encoder scores for the first 4 chunks",
    yaxis=dict(
        title="Cross-encoder delta"
    ),
    xaxis=dict(
        title=""
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'top-4-scores-delta.pdf')

Untersuchung, warum er context_recall, die correctheit und die trustworthiness gesunken sind und ob diese Verschlechterungen zusammenhängen:

In [ ]:
x = np.array(comparing_df['context_recall'].values)
y = np.array(comparing_df['correctness'].values)

mask = ~np.isnan(x) & ~np.isnan(y)
print('Correlation coefficient: context_recall / correctness:')
print(np.corrcoef(x[mask], y[mask])[0][1])

x = np.array(comparing_df['correctness'].values)
y = np.array(comparing_df['trustworthiness'].values)

mask = ~np.isnan(x) & ~np.isnan(y)

print('Correlation coefficient: trustworthiness / correctness:')
print(np.corrcoef(x[mask], y[mask])[0][1])


In [ ]:
context_recall_deltas = comparing_df['context_recall']-baseline_df['context_recall']
correctness_deltas = comparing_df['correctness']-baseline_df['correctness']
trustworthiness_deltas = comparing_df['trustworthiness']-baseline_df['trustworthiness']

x = np.array(context_recall_deltas.values)
y = np.array(correctness_deltas.values)

mask = ~np.isnan(x) & ~np.isnan(y)
print('Correlation coefficient: context_recall delta / correctness delta:')
print(np.corrcoef(x[mask], y[mask])[0][1])

x = np.array(correctness_deltas.values)
y = np.array(trustworthiness_deltas.values)

mask = ~np.isnan(x) & ~np.isnan(y)
print('Correlation coefficient: trustworthiness delta / correctness delta:')
print(np.corrcoef(x[mask], y[mask])[0][1])

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    mode='markers',
    x=context_recall_deltas, 
    y=correctness_deltas)
)
fig.update_layout(
    xaxis_title='context_recall delta', 
    yaxis_title='correctness delta',
    font=dict(
        size=16,
    ))
fig.show()

In [ ]:
comparing_df[comparing_df['context_recall']-baseline_df['context_recall'] < 0]

In [ ]:
comparing_df[(comparing_df['prompt_alignment']-baseline_df['prompt_alignment'] > 0) & (comparing_df['dataset'] == 'fact checking')]

In [ ]:
baseline_df[(comparing_df['prompt_alignment']-baseline_df['prompt_alignment'] > 0) & (baseline_df['dataset'] == 'fact checking')]